# FaceGuard v2 — Presentation Attack Detection (PAD) Fine-Tuning
**Model Architecture:** `MobileNetV3-Small` (RGB Binary Classifier: 1.0 = Bona-fide / Real, 0.0 = Presentation Attack / Spoof)  
**Target Metric:** Minimize Equal Error Rate (EER) pursuant to ISO/IEC 30107-3  
**Output Artifacts:** `mobilenetv3_pad.onnx` & `pad_metrics.json`

---
### Kaggle Instructions:
1. **Accelerator & Internet:**
   - In the right sidebar: **Accelerator** -> **GPU T4 x2** (or **GPU P100**).
   - In the right sidebar: **Internet** -> **Internet On**.
2. **Attach Dataset:**
   - Click **`+ Add Input`** at the top-right.
   - Search for `celeba spoof` (e.g. `marquis03/celeba-spoof-for-face-antispoofing`) or `face anti spoofing` / `liveness detection`.
   - Click the **`+`** button to attach it.
   - *(Optional)* You can also attach your `frad-legacy-crops` dataset for extra genuine faces!
3. **Run All:**
   - Click **Run All** (or `Shift + Enter` cell by cell).
   - In ~3-5 minutes, your trained `mobilenetv3_pad.onnx` will be ready under `/kaggle/working/faceguard/pad/`!


In [ ]:
# --- 1. Dependencies and Environment Setup ---
import os, sys, time, json, random, gc
from pathlib import Path
import numpy as np
from PIL import Image
from sklearn.metrics import roc_curve, auc, roc_auc_score, accuracy_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms, models

# Check optional OpenCV
try:
    import cv2
    print(f"[✓] OpenCV version: {cv2.__version__}")
except Exception as e:
    print(f"[!] Warning on OpenCV: {e} (PIL and Torchvision will be used).")

# Set deterministic seeds for reproducible training
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dev_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f"Using device: {device} ({dev_name})")
if torch.cuda.is_available():
    mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"CUDA Memory: {mem_gb:.2f} GB")

WORK_ROOT = Path("/kaggle/working/faceguard/pad")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print(f"[✓] Working directory created: {WORK_ROOT}")


In [ ]:
# --- 2. Universal Scanner for Attached Kaggle Datasets ---
# Scans all datasets in /kaggle/input and categorizes images into Live (1.0) and Spoof (0.0)
input_root = Path("/kaggle/input")
supported_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print(f"Scanning {input_root} for attached anti-spoofing datasets...")
attached_dirs = [d for d in input_root.iterdir() if d.is_dir()] if input_root.exists() else []
print(f"Found {len(attached_dirs)} attached dataset(s):")
for d in attached_dirs:
    print(f"  📁 {d.name}")

if not attached_dirs:
    raise RuntimeError("No datasets found in /kaggle/input! Please click '+ Add Input' at the top-right and attach CelebA-Spoof or a face anti-spoofing dataset.")

live_samples = []
spoof_samples = []

# Keywords for path classification
live_keywords = {"live", "real", "bonafide", "bona_fide", "genuine", "client", "train_real", "test_real", "val_real"}
spoof_keywords = {"spoof", "fake", "attack", "imposter", "replay", "print", "mask", "screen", "photo", "cut", "train_fake", "test_fake", "val_fake"}

def classify_relative_path(rel_parts):
    for part in reversed(rel_parts):
        p_lower = part.lower()
        for kw in spoof_keywords:
            if kw in p_lower:
                return 0.0
        for kw in live_keywords:
            if kw in p_lower:
                return 1.0
    return None

# Scan each attached dataset
for ds_dir in attached_dirs:
    # Skip code packages
    if "source" in ds_dir.name.lower() or "code" in ds_dir.name.lower():
        print(f"Skipping code package: {ds_dir.name}")
        continue
        
    print(f"\nIndexing: {ds_dir.name} ...")
    ds_live_count = 0
    ds_spoof_count = 0
    
    # Check if this is the FRAD legacy crops dataset (all genuine faces)
    if "frad" in ds_dir.name.lower() and "spoof" not in ds_dir.name.lower() and "attack" not in ds_dir.name.lower():
        for root, _, files in os.walk(ds_dir):
            for f in files:
                if Path(f).suffix.lower() in supported_exts:
                    live_samples.append(Path(root) / f)
                    ds_live_count += 1
        print(f"  -> Added {ds_live_count:,} genuine/live crops from FRAD dataset.")
        continue

    # Otherwise walk the dataset directory
    for root, dirs, files in os.walk(ds_dir):
        img_files = [f for f in files if Path(f).suffix.lower() in supported_exts]
        if not img_files:
            continue
        rel_path = Path(root).relative_to(ds_dir)
        label = classify_relative_path(rel_path.parts)
        for f in img_files:
            full_path = Path(root) / f
            if label == 1.0:
                live_samples.append(full_path)
                ds_live_count += 1
            elif label == 0.0:
                spoof_samples.append(full_path)
                ds_spoof_count += 1
            else:
                fname_lower = f.lower()
                if any(k in fname_lower for k in spoof_keywords):
                    spoof_samples.append(full_path)
                    ds_spoof_count += 1
                elif any(k in fname_lower for k in live_keywords):
                    live_samples.append(full_path)
                    ds_live_count += 1
                    
    print(f"  -> Found {ds_live_count:,} Live images and {ds_spoof_count:,} Spoof images in this dataset.")

print("\n" + "="*50)
print("TOTAL SAMPLES FOUND:")
print(f"  🟢 Live (Bona-fide): {len(live_samples):,} images")
print(f"  🔴 Spoof (Attack):   {len(spoof_samples):,} images")
print("="*50)

if len(live_samples) == 0 or len(spoof_samples) == 0:
    raise RuntimeError(f"Dataset indexing incomplete: Found {len(live_samples)} live and {len(spoof_samples)} spoof images.\n" \
                       "Please make sure you have attached an anti-spoofing dataset (like 'celeba-spoof-for-face-antispoofing' or 'face-anti-spoofing-dataset') using the '+ Add Input' button!")


In [ ]:
# --- 3. Build Balanced Dataset Cohort & DataLoaders ---
class GeneralPADDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples  # List of (path, label)
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            with Image.open(path) as img:
                img = img.convert("RGB")
                if self.transform:
                    img = self.transform(img)
                return img, torch.tensor(label, dtype=torch.float32)
        except Exception:
            fallback = torch.zeros((3, 224, 224), dtype=torch.float32)
            return fallback, torch.tensor(label, dtype=torch.float32)

# Balance classes and cap at 40,000 images (20,000 Live + 20,000 Spoof) for fast, robust training
random.shuffle(live_samples)
random.shuffle(spoof_samples)

max_per_class = min(len(live_samples), len(spoof_samples), 20000)
selected_live = [(p, 1.0) for p in live_samples[:max_per_class]]
selected_spoof = [(p, 0.0) for p in spoof_samples[:max_per_class]]

balanced_cohort = selected_live + selected_spoof
random.shuffle(balanced_cohort)
print(f"[✓] Balanced Training Cohort: {len(balanced_cohort):,} images ({max_per_class:,} Live + {max_per_class:,} Spoof)")

# 80/20 Train/Validation Split
split_idx = int(0.8 * len(balanced_cohort))
train_samples = balanced_cohort[:split_idx]
val_samples = balanced_cohort[split_idx:]
print(f"    - Training split:   {len(train_samples):,} images ({int(split_idx*0.5):,} live, {int(split_idx*0.5):,} spoof)")
print(f"    - Validation split: {len(val_samples):,} images ({int((len(balanced_cohort)-split_idx)*0.5):,} live, {int((len(balanced_cohort)-split_idx)*0.5):,} spoof)")

# Augmentations designed for anti-spoofing (texture-preserving)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Fast pin_memory DataLoader with 2 worker threads
train_loader = DataLoader(GeneralPADDataset(train_samples, train_transform), batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(GeneralPADDataset(val_samples, val_transform), batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print(f"[✓] DataLoaders initialized: {len(train_loader)} train batches, {len(val_loader)} val batches (batch_size=64).")


In [ ]:
# --- 4. MobileNetV3-Small PAD Model Architecture ---
print("Initializing MobileNetV3-Small backbone with ImageNet weights...")
try:
    model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
except Exception as e:
    print(f"Warning: Could not fetch weights directly ({e}). Ensure Internet is toggled ON in Kaggle sidebar.")
    model = models.mobilenet_v3_small(weights=None)

# Replace final classifier layer with a single binary logit output
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, 1)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[✓] Model configured: {total_params:,} total parameters ({trainable_params:,} trainable)")


In [ ]:
# --- 5. GPU Training Loop (5 Epochs with Cosine Annealing & AMP) ---
EPOCHS = 5
best_val_auc = 0.0
best_weights_path = WORK_ROOT / "mobilenetv3_pad_best.pth"

print(f"Starting training on {device} for {EPOCHS} epochs...\n")

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    running_loss, train_correct, train_total = 0.0, 0, 0
    
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        # Mixed precision forward pass for 2-3x speedup on GPU
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(imgs).squeeze(1)
            loss = criterion(logits, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * imgs.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)
        
    scheduler.step()
    train_loss = running_loss / train_total
    train_acc = train_correct / train_total
    
    # Validation Phase
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    all_scores, all_targets = [], []
    
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(imgs).squeeze(1)
                loss = criterion(logits, labels)
                
            val_loss += loss.item() * imgs.size(0)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_scores.extend(probs)
            all_targets.extend(labels.cpu().numpy())
            preds = (probs >= 0.5).astype(float)
            val_correct += (preds == labels.cpu().numpy()).sum()
            val_total += labels.size(0)
            
    val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    val_auc = roc_auc_score(all_targets, all_scores)
    elapsed = time.time() - t0
    
    print(f"Epoch [{epoch}/{EPOCHS}] ({elapsed:.1f}s) | Train Loss: {train_loss:.4f}, Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f}, Acc: {val_acc*100:.2f}%, AUC: {val_auc:.4f}")
    
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), best_weights_path)
        print(f"  ⭐ Saved new best checkpoint -> {best_weights_path.name} (Val AUC: {val_auc:.4f})")

print(f"\n[✓] Training complete! Best Validation AUC: {best_val_auc:.4f} (Saved to {best_weights_path})")


In [ ]:
# --- 6. Threshold Calibration (EER, APCER, BPCER pursuant to ISO/IEC 30107-3) ---
print(f"Loading best checkpoint for threshold calibration: {best_weights_path}...")
model.load_state_dict(torch.load(best_weights_path, map_location=device))
model.eval()

scores, targets = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            probs = torch.sigmoid(model(imgs).squeeze(1)).cpu().numpy()
        scores.extend(probs)
        targets.extend(labels.numpy())

scores = np.array(scores)
targets = np.array(targets)

# ROC Curve (Live = 1, Spoof = 0)
fpr, tpr, thresholds = roc_curve(targets, scores, pos_label=1)
# BPCER = False Presentation Classification Error Rate (Genuine classified as spoof = FNR)
# APCER = Attack Presentation Classification Error Rate (Spoof classified as genuine = FPR)
bpcer = 1.0 - tpr
apcer = fpr

# Find Equal Error Rate (EER) threshold where APCER ≈ BPCER
eer_idx = np.nanargmin(np.abs(apcer - bpcer))
eer_threshold = float(thresholds[eer_idx])
eer_rate = float((apcer[eer_idx] + bpcer[eer_idx]) / 2.0)
final_auc = float(roc_auc_score(targets, scores))

print("\n" + "="*55)
print("📊 PRESENTATION ATTACK DETECTION (PAD) METRICS:")
print(f"  • Calibrated Threshold (τ*): {eer_threshold:.4f}")
print(f"  • Equal Error Rate (EER):    {eer_rate * 100:.2f}%")
print(f"  • APCER at EER:              {apcer[eer_idx] * 100:.2f}%")
print(f"  • BPCER at EER:              {bpcer[eer_idx] * 100:.2f}%")
print(f"  • Area Under ROC Curve (AUC):{final_auc:.4f}")
print("="*55)


In [ ]:
# --- 7. Export to Production ONNX & Parity Verification ---
onnx_output_path = WORK_ROOT / "mobilenetv3_pad.onnx"
metrics_output_path = WORK_ROOT / "pad_metrics.json"

print(f"Exporting model to ONNX format opset=14: {onnx_output_path}...")
model.eval()
cpu_model = model.cpu()
dummy_input = torch.randn(1, 3, 224, 224, dtype=torch.float32)

torch.onnx.export(
    cpu_model,
    dummy_input,
    str(onnx_output_path),
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}}
)

onnx_size_mb = onnx_output_path.stat().st_size / (1024 * 1024)
print(f"[✓] Successfully exported ONNX model! File size: {onnx_size_mb:.2f} MB")

# Test numerical parity with ONNX Runtime
try:
    import onnxruntime as ort
    ort_session = ort.InferenceSession(str(onnx_output_path), providers=["CPUExecutionProvider"])
    test_input = np.random.randn(2, 3, 224, 224).astype(np.float32)
    ort_out = ort_session.run(None, {"input": test_input})[0]
    
    with torch.no_grad():
        torch_out = cpu_model(torch.from_numpy(test_input)).numpy()
        
    max_abs_diff = float(np.max(np.abs(torch_out - ort_out)))
    print(f"[✓] PyTorch vs ONNX Max Absolute Difference: {max_abs_diff:.6e}")
    assert max_abs_diff < 1e-4, "Parity check failed!"
    print("[✓] ONNX Runtime Parity Test: PASSED ✅")
except Exception as e:
    print(f"[!] ONNX Runtime parity check note: {e}")

# Save deployment metrics JSON
metrics_report = {
    "model_architecture": "MobileNetV3-Small",
    "weights_version": "v2_pad_finetuned",
    "input_resolution": [224, 224, 3],
    "input_mean": [0.485, 0.456, 0.406],
    "input_std": [0.229, 0.224, 0.225],
    "calibrated_threshold": round(eer_threshold, 4),
    "eer": round(eer_rate, 4),
    "auc": round(final_auc, 4),
    "onnx_filename": onnx_output_path.name,
    "onnx_size_bytes": onnx_output_path.stat().st_size,
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
}

with open(metrics_output_path, "w") as f:
    json.dump(metrics_report, f, indent=2)
print(f"[✓] Deployment metrics saved to: {metrics_output_path}")

print(f"\n{'='*60}")
print("🎉 ANTI-SPOOFING MODEL READY FOR DOWNLOAD!")
print(f"1. Download '{onnx_output_path.name}' from the Kaggle Output tab.")
print(f"2. Download '{metrics_output_path.name}' from the Kaggle Output tab.")
print(f"3. Place both files in your local project at: models/releases/")
print(f"{'='*60}")
